In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
# print(stoi)
itos = {v: k for k, v in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [32]:
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
    # print(w)
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]
        # print(''.join(itos[i] for i in context),' --> ', ch)
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix] # crop and append
X = torch.tensor(X)
Y = torch.tensor(Y)

In [33]:
X.shape, X.dtype, Y.shape, Y. dtype 

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [34]:
C = torch.randn((27, 2))

In [35]:
C[X].shape

torch.Size([32, 3, 2])

In [36]:
# example on how pytorch indexing works. C[X]
eg = torch.tensor([[0, 0, 1],[0, 1, 1]])
print('eg.shape:  ', eg.shape)
print('C[eg].shape', C[eg].shape)
C[eg]

eg.shape:   torch.Size([2, 3])
C[eg].shape torch.Size([2, 3, 2])


tensor([[[-0.5261,  0.9633],
         [-0.5261,  0.9633],
         [-0.8873,  0.7180]],

        [[-0.5261,  0.9633],
         [-0.8873,  0.7180],
         [-0.8873,  0.7180]]])

In [37]:
C[X][13, 2]

tensor([-0.8873,  0.7180])

In [38]:
X[13, 2]

tensor(1)

In [39]:
# C[X][13, 2] = C[X[13, 2]]
C[1]

tensor([-0.8873,  0.7180])

In [40]:
# embedd all of the integers in X
emb =  C[X]
emb.shape

torch.Size([32, 3, 2])

In [41]:
# implementing the hidden layer

# no. of inputs to the hidden layer is 6 with our current settings. Why?
# because we have 2 dimensional embeddings of 3 units of look-up tables.(3 previous chars(context_length) to predict the 4th char)
W1 = torch.randn((6, 100))
b1 = torch.rand(100)

# emb @ W1 + b1  # work won't 

In [42]:
# We need to transform emb to (32, 6) from (32, 3, 2) to perform dot product with W1 (6, 100 )

# emb[:, 0, :]
emb[:, 0, :].shape
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1 )

torch.Size([32, 2])

In [43]:
a = torch.tensor([[[1, 2], [3, 4], [5, 6]],[[7, 8], [9, 10], [11, 12]], [[13, 14], [15, 16], [17, 18]]])
print(a)
print(a.shape)
# print(torch.unbind(a, 1))
print(torch.cat(torch.unbind(a, 1), 1))
print(torch.cat(torch.unbind(a, 1), 1).shape)

# torch.cat(torch.unbind(emb, 1), 1) == torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1 )

tensor([[[ 1,  2],
         [ 3,  4],
         [ 5,  6]],

        [[ 7,  8],
         [ 9, 10],
         [11, 12]],

        [[13, 14],
         [15, 16],
         [17, 18]]])
torch.Size([3, 3, 2])
tensor([[ 1,  2,  3,  4,  5,  6],
        [ 7,  8,  9, 10, 11, 12],
        [13, 14, 15, 16, 17, 18]])
torch.Size([3, 6])


In [44]:
# this is one way to transform the input to (32, 6) tensor using torch.cat(torch.unbind(emb,1),1)
# but turns out there is a different way which is a much efficient way than this! (explained in next cell)

# torch.unbind(emb, 1)
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [45]:
# calling .view() is extremely effiecient in pytorch. Reason for that is that in each tensor, 
# there's something called the underlying storage.

# tensor.view() works as long as the product of the dims is equal to the size of the tensor. 
a = torch.arange(18)
print(a)
a.view(2, 9)
# a.view(9, 2)
# a.view(3, 3, 2)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])


tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [46]:
# the storage is just the numbers always stored as a one-dimensional vector and 
# this is how a tensor is represented in the computer memory.
# so when view() is manipulating how this 1-d sequence is interpreted as a n-dimensional tensor.
# No memory is changed, copied, moved or deleted while we call tensor.view(), 
# instead some storage attributes storage offsets, strides and shapes are amnipulated.
# For more information:  (https://blog.ezyang.com/2019/05/pytorch-internals/)

In [47]:
emb.shape

torch.Size([32, 3, 2])

In [48]:
# emb.view(32, 6) == torch.cat(torch.unbind(emb, 1), 1)
# emb.view(-1, 6) # pytorch will infer the value at -1 automatically!

h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # output of the hidden layer
h

tensor([[ 0.4693, -0.8326,  0.9620,  ...,  0.8660,  0.7146,  0.9999],
        [ 0.3325, -0.3915,  0.9686,  ...,  0.8374,  0.7756,  0.9999],
        [-0.5742,  0.9998,  0.9962,  ...,  0.0428,  0.6191,  0.9930],
        ...,
        [-0.9881, -0.9997,  0.5554,  ...,  0.9834, -0.9180,  1.0000],
        [ 0.5210, -0.9847,  0.6527,  ...,  0.8400,  0.4409,  0.9831],
        [ 0.5081, -0.8227,  0.8970,  ...,  0.9645,  0.9773,  0.9741]])

In [49]:
h.shape

torch.Size([32, 100])

In [50]:
print((emb.view(-1, 6) @ W1).shape)
print(b1.shape)
# 32, 100   (emb.view(-1, 6) @ W1).shape
# 1,  100    updated b1.shape from broadcasting

# broadcasting process, step by step.
# step 1: Align to the right when there is shape mismatch
# step 2: Create a fake dimension(1) to the left. -> (1, 100)
# step 3: Copy the (1, 100) row vector n times to match the other tensor(n=32 in this case)
# step 4: Complete element-wise addition:  (emb.view(-1, 6) @ W1) + b1

torch.Size([32, 100])
torch.Size([100])


In [51]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [52]:
logits = h @ W2 + b2                   # log-counts

In [53]:
logits.shape

torch.Size([32, 27])

In [54]:
counts = logits.exp()                  # exponentiate logits to get "counts"

In [55]:
prob = counts / counts.sum(dim=1, keepdims=True)
prob.shape
# prob[1].sum()

torch.Size([32, 27])

In [56]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [57]:
# calculate loss: negative log likelihood
# loss = -nll(torch.arange(32) is just for iterating through all training data)
# and 'Y' extract the relevant prob for each training data (Y: ground truth)
loss = -prob[torch.arange(32), Y].log().mean() 
loss                                           

tensor(17.3469)

In [58]:
# now made respectable!----------

In [59]:
X.shape, Y.shape       # dataset

(torch.Size([32, 3]), torch.Size([32]))

In [60]:
g = torch.Generator().manual_seed(2147483647)  # for reproducability

C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)

W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [61]:
sum(p.nelement() for p in parameters) # no. of parameters in total

3481

In [62]:
for p in parameters:
    p.requires_grad = True

In [69]:
for _ in range(1000):
    # forward pass
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)            # h.shape = (32, 100)
    logits = h @ W2 + b2                                 # logits.shape = (32, 27)
    # counts = logits.exp()
    # prob = counts / counts.sum(dim=1, keepdims=True)
    # loss = -prob[torch.arange(32), Y].log().mean()
    loss = F.cross_entropy(logits, Y)
    
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # update
    for p in parameters:
        p.data += -0.1 * p.grad
    
print(loss.item())

0.25336122512817383
